# 04 · Hiệu chuẩn, ngưỡng, lợi ích lâm sàng và quy trình review (phụ trách: Hà)

**Đề:** *"Evaluate … calibration"* + *"Introduce human review for high-risk or low-confidence predictions"*.
Mã nguồn: `trustworthy_ai/calibration.py`, `trustworthy_ai/utility.py`, `trustworthy_ai/review.py`, `backend/app/routers/reviews.py`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

In [ ]:
R = load_results(); ch = R['chosen_model']; print('Mô hình:', ch, '| hiệu chuẩn:', R['calibration'][ch]['method'])

## 1. So sánh phương pháp hiệu chuẩn (chọn theo Brier trên dự đoán ngoài fold)

In [ ]:
pd.DataFrame(R['calibration'][ch]['comparison']).T

## 2. Ngưỡng LOW / MEDIUM / HIGH

In [ ]:
{k: v for k, v in R['thresholds'].items() if not isinstance(v, dict)}, R['dev_risk_level_mortality']

## 3. Decision Curve Analysis và bảng điểm cắt

In [ ]:
print(R['dca_dev']['summary']); pd.DataFrame(R['operating_dev'])

## 4. Luật review: khối lượng hàng đợi và hiệu quả

In [ ]:
rv = R['review_dev']; print({k: rv[k] for k in ('flagged_share', 'deaths_captured_share', 'death_rate_in_queue', 'death_rate_not_flagged')}); pd.DataFrame(rv['by_reason']).T

## 5. Bằng chứng luật "độ tin cậy thấp" có tác dụng + risk–coverage

In [ ]:
R['confidence_validation_dev']

In [ ]:
pd.DataFrame(R['risk_coverage_dev'])

## 6. Quy trình trên hệ thống (gọi API thật bằng TestClient)

In [ ]:
os.environ.setdefault("DATABASE_URL", "sqlite:///artifacts/notebook_demo.db")
from fastapi.testclient import TestClient
from backend.app.main import app
with TestClient(app) as c:
    case = next(x for x in c.get(f"/api/v1/cases/{TASK}").json() if x["case_id"].endswith("sparse"))
    r = c.post(f"/api/v1/predict/{TASK}", json={"case_id": case["case_id"], "features": case["features"]}).json()
    print("Cờ:", r["review"])
    rid = r["review"]["review_id"]
    print(c.patch(f"/api/v1/reviews/{rid}", json={"status": "REVIEWED", "reviewer": "ha"}).status_code, "← nhảy cóc bị chặn (409)")
    c.patch(f"/api/v1/reviews/{rid}", json={"status": "IN_REVIEW", "reviewer": "ha"})
    done = c.patch(f"/api/v1/reviews/{rid}", json={"status": "REVIEWED", "reviewer": "ha", "decision": "AGREE"}).json()
    print([(e["old_status"], e["new_status"], e["actor"]) for e in done["audit"]])

## Câu hỏi tự kiểm tra
1. O/E = 0.8 nghĩa là gì? Slope < 1 nghĩa là gì?
2. Vì sao ngưỡng HIGH chọn theo F1 còn MEDIUM theo độ nhạy ≥ 80%?
3. Lợi ích ròng (net benefit) tính thế nào? Khi nào mô hình "vô ích"?
4. 7 lý do gắn cờ là gì? Lý do nào là "nguy cơ cao", lý do nào là "độ tin cậy thấp"?